In [141]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math

In [142]:
def plot_images(images, convert=None, titles=None, cols=3, figsize=(15, 10)):
    # Number of images
    num_images = len(images)
    
    # Calculate number of rows
    rows = math.ceil(num_images / cols)
    
    # Create the subplot grid
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.ravel()  # Flatten the 2D grid of axes to iterate over
    
    # Loop through each image and corresponding axis
    for i in range(rows * cols):
        if i < num_images:
            if (convert == "BGR2RGB"):
                images[i] = cv2.cvtColor(images[i], cv2.COLOR_BGR2RGB)
                
            # Display image
            axes[i].imshow(images[i], cmap='gray' if len(images[i].shape) == 2 else None)
            
            # Add title if provided
            if titles:
                axes[i].set_title(titles[i], fontsize=12)
        else:
            # Hide any unused subplot
            axes[i].axis('off')
        
        # Remove axes for clarity
        axes[i].axis('off')
    
    # Adjust layout for better spacing
    plt.tight_layout()
    plt.show()

In [143]:
def extract_roi(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Threshold the image to create a binary image (non-black areas will be white)
    _, thresh = cv2.threshold(gray, 1, 255, cv2.THRESH_BINARY)
    
    # Find contours of the non-black areas
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Find the bounding box of the largest contour (which should be the region of interest)
    x, y, w, h = cv2.boundingRect(contours[0])
    
    return x, y, w, h


# Crop the image to the bounding box (ROI)
def crop_image(image, x, y, w, h):
    return image[y:y+h, x:x+w]



In [144]:
def manually_align_images(source_color, source_gray, target_color, target_gray):
    """
    Manually aligns source and target images by allowing the user to select corresponding points.
    
    Args:
        source_color (ndarray): Source image in color.
        source_gray (ndarray): Source image in grayscale.
        target_color (ndarray): Target image in color.
        target_gray (ndarray): Target image in grayscale.
    
    Returns:
        aligned_source (ndarray): The aligned source image.
        matched_img (ndarray): Visualization of matched points.
    """

    def select_points(image, window_name):
        """
        Allows the user to manually select points on an image by clicking.
        Press Enter to finish the selection.
        
        Args:
            image (ndarray): Image to display for point selection.
            window_name (str): Name of the OpenCV window.

        Returns:
            points (list): List of selected points.
        """
        points = []
        drawing = [image.copy()]  # Use a list to store the updated image (mutable)

        def draw_and_select(event, x, y, flags, param):
            if event == cv2.EVENT_LBUTTONDOWN:
                points.append((x, y))
                # Draw the clicked point and lines connecting points
                for i, pt in enumerate(points):
                    cv2.circle(drawing[0], pt, 5, (0, 255, 0), -1)  # Draw point

                cv2.imshow(window_name, drawing[0])  # Update the image display

        # Display the image for point selection
        cv2.imshow(window_name, image)
        cv2.setMouseCallback(window_name, draw_and_select)

        # Wait for the user to finish point selection by pressing Enter
        while True:
            key = cv2.waitKey(1) & 0xFF
            if key == 13:  # Enter key
                break

        cv2.destroyWindow(window_name)
        return points

    # Select points for the source image
    source_points = select_points(source_color, "Select Points - Source Image")
    print(f"Selected points from source: {source_points}")

    # Select points for the target image
    target_points = select_points(target_color, "Select Points - Target Image")
    print(f"Selected points from target: {target_points}")

    # Convert points to NumPy arrays
    src_pts = np.float32(source_points).reshape(-1, 1, 2)
    tgt_pts = np.float32(target_points).reshape(-1, 1, 2)

    # Compute the homography matrix
    H, mask = cv2.findHomography(src_pts, tgt_pts, cv2.RANSAC, 5.0)

    # Warp the source image
    height, width = target_gray.shape
    aligned_source = cv2.warpPerspective(source_color, H, (width, height))

    # Create a combined image for SIFT-style visualization
    combined_width = source_color.shape[1] + target_color.shape[1]
    combined_height = max(source_color.shape[0], target_color.shape[0])
    combined_img = np.zeros((combined_height, combined_width, 3), dtype=np.uint8)

    # Place source and target images side by side
    combined_img[:source_color.shape[0], :source_color.shape[1]] = source_color
    combined_img[:target_color.shape[0], source_color.shape[1]:] = target_color

    # Draw the points and connecting lines
    for (sx, sy), (tx, ty) in zip(source_points, target_points):
        # Draw source points (green dots)
        cv2.circle(combined_img, (int(sx), int(sy)), 5, (0, 255, 0), -1)

        # Draw target points (red dots)
        cv2.circle(combined_img, (int(tx) + source_color.shape[1], int(ty)), 5, (0, 0, 255), -1)

        # Draw connecting lines (blue)
        cv2.line(combined_img, (int(sx), int(sy)), (int(tx) + source_color.shape[1], int(ty)), (255, 0, 0), 2)

    return aligned_source, combined_img

In [145]:
def automatically_align_images(source_color, source_gray, target_color, target_gray):
    target_height, target_width = target_gray.shape

    # Extract ROI from target image (remove the black box if needed)
    x, y, w, h = extract_roi(source_color)

    # Crop the images based on ROI
    source_color = crop_image(source_color, x, y, w, h)
    source_gray = crop_image(source_gray, x, y, w, h)

    # Resize back up to target image size
    source_color = cv2.resize(source_color, (target_width, target_height))
    source_gray = cv2.resize(source_gray, (target_width, target_height))

    orb = cv2.ORB_create()

    # Detect keypoints and descriptors
    source_kp, source_des = orb.detectAndCompute(source_gray, None)
    target_kp, target_des = orb.detectAndCompute(target_gray, None)

    # Match descriptors using BFMatcher
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(source_des, target_des)
    
    # Sort matches by distance (best matches first)
    matches = sorted(matches, key=lambda x: x.distance)

    # Extract the corresponding points
    source_pts = np.float32([source_kp[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    target_pts = np.float32([target_kp[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    # Compute the homography
    H, mask = cv2.findHomography(source_pts, target_pts, cv2.RANSAC, 5.0)

    # Warp the source image to align with the target
    aligned_source = cv2.warpPerspective(source_color, H, (target_width, target_height))

    # Draw the matches
    matched_img = cv2.drawMatches(source_color, source_kp, target_color, target_kp, matches, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    return aligned_source, matched_img

In [146]:
def align_images(source, target, selection='automatic'):
    source_color = cv2.imread(source)
    target_color = cv2.imread(target)

    source_gray = cv2.imread(source, cv2.IMREAD_GRAYSCALE)
    target_gray = cv2.imread(target, cv2.IMREAD_GRAYSCALE)

    # Feature detection and matching
    if selection == 'automatic':
        aligned_source, matched_img = automatically_align_images(source_color, source_gray, target_color, target_gray)
    elif selection == 'manual':
        aligned_source, matched_img = manually_align_images(source_color, source_gray, target_color, target_gray)

    # Display the results
    plt.figure(figsize=(12, 6))

    # Display source and target images with points
    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(source_color, cv2.COLOR_BGR2RGB))
    plt.title('source')
    plt.axis("off")
    
    plt.subplot(1, 3, 2)
    plt.imshow(cv2.cvtColor(target_color, cv2.COLOR_BGR2RGB))
    plt.title('target')
    plt.axis("off")
    
    plt.subplot(1, 3, 3)
    plt.imshow(cv2.cvtColor(matched_img, cv2.COLOR_BGR2RGB))
    plt.title('Matched Points')
    plt.axis("off")
    plt.show()

    # Display the aligned source image
    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(aligned_source, cv2.COLOR_BGR2RGB))
    plt.title('Transformed/aligned source')
    plt.axis("off")
    plt.show()

In [ ]:
# align_images('Task-3/frail.png', 'Task-3/frail_reference.webp', 'automatic')
# align_images('Task-3/zebra_crossing1.png', 'Task-3/zebra_crossing_reference.jpg', 'automatic')
align_images('Task-3/zebra_crossing1.png', 'Task-3/zebra_crossing_reference.jpg', 'manual')